# Encrypted TextGen in Colab

Full TextGen interface with browser vault unlock. Select **Runtime → Change runtime type → GPU**, then run cells 1–4 in order. Passwords and chats belong in the embedded interface, never in notebook cells.

This notebook uses [NulAsh/textgen](https://github.com/NulAsh/textgen). It replaces Landlock with a verified read-only filesystem sandbox and a dedicated unprivileged worker. If the runtime cannot enforce it, startup stops before decrypting anything. The local desktop launcher still uses Landlock.

Google's [Colab usage restrictions](https://research.google.com/colaboratory/faq.html#limitations-and-restrictions) still apply, including the free-tier web-interface restriction. This notebook adds no keepalive or restriction-evasion mechanisms. Google processes plaintext in RAM during inference; the vault protects saved files, not content from the runtime provider.

**Status:** experimental Colab integration. Automated Linux tests cover the sandbox, gateway and CPU inference. Colab's live browser proxy, DriveFS and GPU execution still require testing in your session.

In [ ]:
#@title 1. Install the fork and verify write isolation
import os
import shutil
import subprocess
import sys
from pathlib import Path

if os.geteuid() != 0:
    raise RuntimeError("This Colab launcher requires the standard root notebook runtime.")

TEXTGEN_ROOT = Path("/content/textgen-encrypted")
TEXTGEN_ENV = Path("/content/textgen-encrypted-env")
TEXTGEN_PYTHON = TEXTGEN_ENV / "bin/python"

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "bubblewrap", "libseccomp2"], check=True)
if not TEXTGEN_ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/NulAsh/textgen.git", str(TEXTGEN_ROOT)], check=True)
else:
    remote = subprocess.check_output(["git", "-C", str(TEXTGEN_ROOT), "remote", "get-url", "origin"], text=True).strip()
    if remote.removesuffix(".git") != "https://github.com/NulAsh/textgen":
        raise RuntimeError("This folder is not the encrypted fork. Choose a separate folder.")
    subprocess.run(["git", "-C", str(TEXTGEN_ROOT), "pull", "--ff-only"], check=True)

if not TEXTGEN_PYTHON.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(TEXTGEN_ENV)], check=True)
subprocess.run([str(TEXTGEN_PYTHON), "-m", "pip", "install", "-q", "-r", str(TEXTGEN_ROOT / "requirements/vault.txt")], check=True)
subprocess.run([str(TEXTGEN_PYTHON), "-B", "-m", "modules.vault_colab"], cwd=TEXTGEN_ROOT, check=True)
print("Write isolation verified. Continue to cell 2.")

## Model installation
Cell 2 installs the CUDA version in an isolated Python environment. Optionally enter an **HTTPS URL to a public GGUF file**. Hugging Face file-page links (`/blob/`) are converted to download links (`/resolve/`). The downloaded GGUF header and metadata are checked before installation; an invalid existing file is replaced only after a new download passes those checks.

You can also upload an existing GGUF into `/content/textgen-models` using Colab's Files panel, then leave the URL blank and run cell 2 to check uploaded files. Select and load the model in TextGen's Model tab after unlocking. For split models, download every shard and select the first one. Metadata checks do not verify tensor contents or checksums.

In [ ]:
#@title 2. Install CUDA dependencies and optionally download a GGUF
model_url = "" #@param {type:"string"}

TEXTGEN_MODELS = Path("/content/textgen-models")
TEXTGEN_MODELS.mkdir(exist_ok=True)
subprocess.run([str(TEXTGEN_PYTHON), "-m", "pip", "install", "-q", "-r", str(TEXTGEN_ROOT / "requirements/portable/requirements.txt")], check=True)
subprocess.run([str(TEXTGEN_PYTHON), "-B", "-m", "modules.gguf_download",
                "--model-dir", str(TEXTGEN_MODELS), "--url", model_url.strip()],
               cwd=TEXTGEN_ROOT, check=True)

## Persistent encrypted data
Cell 3 mounts Google Drive and stores encrypted snapshots in `MyDrive/TextGenEncrypted`. The active SQLite vault stays on Colab's local disk; the live database/WAL are not placed on DriveFS. The gateway writes a complete encrypted snapshot roughly every minute after changes and on **Save and lock**. It keeps the latest and previous snapshots.

For your existing desktop vault, use **Encrypted backup** on your laptop, then upload that file to the Drive folder as `vault-latest.sqlite3` **before** running cell 3 in a fresh runtime. Existing local runtime data is never overwritten automatically. Password changes do not change the password of older backups.

Colab can terminate a session abruptly. Changes after the last completed Drive snapshot may be lost. The toolbar shows the most recent backup time. Keep a downloaded encrypted backup for important work.

In [ ]:
#@title 3. Connect encrypted backups to Google Drive
from google.colab import drive

drive.mount("/content/drive")
TEXTGEN_BACKUPS = Path("/content/drive/MyDrive/TextGenEncrypted")
TEXTGEN_BACKUPS.mkdir(parents=True, exist_ok=True)
TEXTGEN_VAULT = Path("/content/textgen-encrypted-vault.sqlite3")
latest = TEXTGEN_BACKUPS / "vault-latest.sqlite3"
if not TEXTGEN_VAULT.exists() and latest.is_file():
    temporary = TEXTGEN_VAULT.with_suffix(".restoring")
    shutil.copyfile(latest, temporary)
    temporary.chmod(0o600)
    temporary.replace(TEXTGEN_VAULT)
    print("Encrypted snapshot restored. Use its existing password in the interface.")
elif TEXTGEN_VAULT.exists():
    print("Keeping the current runtime's vault.")
else:
    print("A new vault will be created when you choose a password in the interface.")

In [ ]:
#@title 4. Open the full encrypted TextGen interface
context_size = 8192 #@param {type:"integer"}

import secrets
import signal
import socket
import tempfile
import time
import urllib.parse
import urllib.request
from google.colab import output

TEXTGEN_PORT = 7860
previous_process = globals().get("vault_process")
if previous_process is not None and previous_process.poll() is None:
    raise RuntimeError("The vault is already running. Use cell 5 to stop it before restarting.")

with socket.socket() as port_check:
    port_check.bind(("127.0.0.1", TEXTGEN_PORT))

proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({TEXTGEN_PORT}, {{cache: false}})")
parsed = urllib.parse.urlsplit(proxy_url)
if parsed.scheme != "https" or parsed.path not in ("", "/"):
    raise RuntimeError("Colab returned an unsupported proxy URL.")
proxy_origin = urllib.parse.urlunsplit((parsed.scheme, parsed.netloc, "", "", ""))
command = [str(TEXTGEN_PYTHON), "-B", "server.py", "--portable", "--vault-isolation", "colab",
           "--vault-colab-origin", proxy_origin, "--vault", str(TEXTGEN_VAULT),
           "--vault-backup-dir", str(TEXTGEN_BACKUPS), "--model-dir", str(TEXTGEN_MODELS),
           "--ctx-size", str(context_size), "--listen-port", str(TEXTGEN_PORT)]
# Pin the proxy Host once using a random capability, before any vault unlock.
colab_bootstrap = secrets.token_urlsafe(32)
launch_env = dict(os.environ, TEXTGEN_COLAB_BOOTSTRAP_TOKEN=colab_bootstrap)
log = tempfile.NamedTemporaryFile(prefix="textgen-colab-launch-", suffix=".log", dir="/dev/shm", delete=False)
launch_log = Path(log.name)
try:
    vault_process = subprocess.Popen(command, cwd=TEXTGEN_ROOT, stdin=subprocess.DEVNULL, stdout=log, stderr=log, env=launch_env)
finally:
    log.close()

deadline = time.monotonic() + 90
while True:
    if vault_process.poll() is not None:
        # The interface has not been displayed/unlocked yet. Startup diagnostics
        # contain no password or decrypted user content.
        raise RuntimeError("Vault startup failed:\n" + launch_log.read_text(errors="replace")[-4000:])
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{TEXTGEN_PORT}/", timeout=1) as response:
            if response.status == 200:
                break
    except OSError:
        pass
    if time.monotonic() >= deadline:
        vault_process.send_signal(signal.SIGINT)
        raise RuntimeError("Vault startup timed out. Check the startup log in RAM: " + str(launch_log))
    time.sleep(0.25)

# Use the exact URL configured above, even if the helper obtains another URL.
bootstrap_url = proxy_origin + "/?" + urllib.parse.urlencode({"vault-bootstrap": colab_bootstrap})
output.serve_kernel_port_as_iframe(TEXTGEN_PORT, path=bootstrap_url, height=950, cache_in_notebook=False)
del colab_bootstrap, launch_env, bootstrap_url
print("Unlock inside the interface, then select your GGUF in the Model tab. Use Save and lock when finished.")

In [ ]:
#@title 5. Stop after saving
import hashlib

# Stop only after Save and lock has completed, so a failed Drive backup
# cannot shut down the supervisor that retains unsaved RAM data.
workspace_id = hashlib.sha256(os.fsencode(TEXTGEN_VAULT.resolve())).hexdigest()[:24]
workspace = Path("/dev/shm") / f"textgen-vault-{os.getuid()}-{workspace_id}"
if workspace.exists():
    raise RuntimeError("Use Save and lock in the interface first and wait for it to finish. If saving failed, leave this runtime running and correct the storage problem before retrying.")
if "vault_process" in globals() and vault_process.poll() is None:
    vault_process.send_signal(signal.SIGINT)
    try:
        vault_process.wait(timeout=60)
    except subprocess.TimeoutExpired:
        raise RuntimeError("Shutdown has not finished. Leave the runtime running and inspect the startup log in RAM.")
    print("Gateway stopped. Your encrypted snapshots are in the Drive folder.")
else:
    print("The gateway is already stopped.")
